# Example: Let's Decompose a Synthetic Data Cloud using Singular Value Decomposition (SVD)
In this example, we generate a synthetic dataset that simulates a cloud of points in a two-dimensional space. We'll then decompose this dataset using Singular Value Decomposition (SVD) to identify the principal directions of variance in the data. 

What does SVD tell us about the structure of the data? How can we interpret the singular values and vectors in the context of this synthetic dataset? Let's find out.


> __Learning Objectives:__
> 
> By the end of this example, you should be able to:
> 
> * __Compute empirical covariance matrices from data:__ Calculate sample covariance matrices using centered data and verify results against standard implementations
> * __Apply power iteration to find dominant eigenpairs:__ Use the power iteration method to estimate the largest eigenvalue and associated eigenvector of a matrix
> * __Interpret eigendecomposition in data analysis:__ Explain how dominant eigenvectors represent directions of maximum variance in multidimensional datasets


Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

### Data
Let's generate a synthetic dataset that simulates a cloud of points in a two-dimensional space. We'll create points that are normally distributed around the origin with some added noise. 

> __Synthetic Data Model__
> 
> We generate points from a bivariate normal distribution with mean vector $\mathbf{\mu} \in \mathbb{R}^{2}$ and covariance matrix $\mathbf{\Sigma} \in \mathbb{R}^{2 \times 2}$. The correlation parameter $\rho \in [-1, 1]$ controls the linear relationship between the two dimensions. When $\rho = 0$, the dimensions are uncorrelated; when $\rho < 0$, they are negatively correlated; and when $\rho > 0$, they are positively correlated. The covariance matrix structure $\mathbf{\Sigma} = [1.0\, \rho; \rho\, 1.0]$ ensures unit variance in each dimension while introducing the specified correlation.

We'll save the data in the `X::Array{Float64,2}` variable, where each row represents a data point in 2D space.

In [2]:
X, Σ = let

    # initialize -
    number_of_points = 25000; # TODO: number of data points to generate, update as needed

    # specify distribution parameters
    μ = [0.1, 2.0]; # mean of the distribution
    ρ = -0.5; # create some correlation between dimensions
    Σ = [1.0 ρ; ρ 1.0]; # TODO: covariance matrix, update as needed
    d = MvNormal(μ, Σ); # define the multivariate normal distribution

    # generate synthetic data
    data = rand(d, number_of_points) |> transpose |> Matrix; # generate points and transpose to get points as rows

    (data, Σ) # return the data and covariance matrix
end

([0.5937322358269439 0.05536015768131075; 0.34138864307568284 2.2360577857309623; … ; 1.2499852951626897 2.0053637017219175; -1.45498441417135 2.1432486125582817], [1.0 -0.5; -0.5 1.0])

Let's form the centered data matrix $\tilde{\mathbf{X}}$ by subtracting the mean from each row of the data matrix $\mathbf{X}$. We store the centered data in the `X_centered::Array{Float64,2}` variable:

In [3]:
X_centered = let 
    r, c = size(X)
    m = mean(X, dims=1) |> vec # mean for each dimension
    ones_vector = ones(r)
    X̃ = X .- ⊗(ones_vector, m);
end

25000×2 Matrix{Float64}:
  0.500794   -1.93814
  0.24845     0.242559
  0.482997    0.709998
  1.05845     0.29072
 -0.368585    1.38305
  0.876444    0.199612
 -0.125193    0.479727
  1.50117    -0.79042
 -0.46266     0.130878
  0.0369334   0.113227
  ⋮          
  1.92663    -0.819272
  0.285452    0.0378735
 -1.4762      2.56798
  0.220283   -0.107447
  0.764886    0.273334
 -2.40463     2.44052
  0.621184   -0.158157
  1.15705     0.0118652
 -1.54792     0.14975

___

## Task 1: Perform SVD on the Synthetic Data Cloud
In this task, we'll perform Singular Value Decomposition (SVD) on the synthetic dataset to identify the principal directions of variance. We'll make use of [the `svd(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.svd) from Julia's standard library to do this computation.

> __How is this different from PCA via Eigendecomposition?__ While PCA typically involves computing the covariance matrix and then performing eigendecomposition, SVD directly decomposes the data matrix itself. This approach can be more efficient and numerically stable, especially for large datasets.

We'll use the [`svd(...)` method exported from the `LinearAlgebra.jl` package](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.SVD) to compute the singular value decomposition of the synthetic dataset `X::Array{Float64,2}`.  This produces the `U`, `S`, and `V` matrices, holding the left singular vectors, the singular values, and the right singular vectors, respectively.

In [4]:
U,S,V = svd(X_centered); # perform SVD on centered data matrix

What's in the `U::Array{Float64,2}`, `S::Array{Float64,1}`, and `V::Array{Float64,2}` matrices? How do they relate to the structure of our synthetic dataset? 

Let's look at each, starting with the `U::Array{Float64,2}` matrix.

> __What is in the `U::Array{Float64,2}` matrix?__ The `U` matrix contains the left singular vectors of the data matrix `X`. Each column of `U` represents a direction in the original data space along which the data varies. The left singular vectors are orthonormal, meaning they are unit vectors that are mutually perpendicular.

The `U::Array{Float64,2}` matrix will have dimension of number of samples by the number of features (2 in this case). Each column corresponds to a principal direction in the __data space__:

In [5]:
U

25000×2 Matrix{Float64}:
 -0.00890825    0.00905025
 -2.40602e-5   -0.00308423
  0.000823678  -0.00749582
 -0.00281344   -0.00846806
  0.00639797   -0.00638784
 -0.00247973   -0.00675327
  0.00220949   -0.00223236
 -0.00838069   -0.00444434
  0.00217142    0.00207885
  0.000278123  -0.000943909
  ⋮            
 -0.0100435    -0.00693158
 -0.0009067    -0.00202878
  0.0147781    -0.00689375
 -0.00119862   -0.000705884
 -0.00180223   -0.00651724
  0.0177115    -0.000268272
 -0.0028513    -0.00290162
 -0.00419228   -0.00733241
  0.00621313    0.00876763

How about the `V::Array{Float64,2}` matrix?

> __What is in the `V::Array{Float64,2}` matrix?__ The `V` matrix contains the right singular vectors of the data matrix `X`. Each column of `V` represents a direction in the feature space along which the data varies. Similar to `U`, the right singular vectors are also orthonormal.

The `V::Array{Float64,2}` matrix will have dimension of number of features by number of features (2 in this case). Each column corresponds to a principal direction in the __feature space__:

In [6]:
V

2×2 adjoint(::Matrix{Float64}) with eltype Float64:
 -0.708101  -0.706111
  0.706111  -0.708101

Finally, what about the `S::Array{Float64,1}` vector?

> __What is in the `S::Array{Float64,1}` vector?__ The `S` vector contains the singular values of the data matrix `X`. These values are non-negative and are typically arranged in descending order. Each singular value indicates the amount of variance captured along the corresponding singular vector directions in `U` and `V`.

What are the singular values?

In [7]:
S

2-element Vector{Float64}:
 193.43342164461072
 112.56946322339132


### Relationship between Singular Values and Eigenvalues
An obvious question to ask is: How are singular values related to eigenvalues?

> __Relationship between Singular Values and Eigenvalues:__ The singular values in `S` are related to the eigenvalues of the covariance matrix of the data. Specifically, if we compute the covariance matrix $\mathbf{C} = \frac{1}{n-1} \mathbf{X}^T \mathbf{X}$, where $n$ is the number of samples, the eigenvalues of $\mathbf{C}$ are equal to the squares of the singular values divided by $(n-1)$. Thus, if $\sigma_i$ is a singular value, then the corresponding eigenvalue $\lambda_i$ of the covariance matrix is given by $\lambda_i = \frac{\sigma_i^2}{n-1}$.

Let's test out this claim. We can estimate the covariance matrix from the synthetic dataset `X::Array{Float64,2}` using the [`cov(...)` function from the `Statistics.jl` package](https://docs.julialang.org/en/v1/stdlib/Statistics/#Statistics.cov). Then, we can compute the eigenvalues of the covariance matrix using the [`eigen(...)` function from the `LinearAlgebra.jl` package](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen).

First, let's compute the empirical covariance matrix $\hat{\mathbf{\Sigma}}$ and store it in the `Σ̂::Array{Float64,2}` variable:

In [8]:
Σ̂ = let 

    # initialize -
    (r,c) = size(X_centered)
    Σ = (1/(r-1)) * (X_centered' * X_centered)
    Σ; # return the empirical covariance matrix
end

2×2 Matrix{Float64}:
  1.0032   -0.49491
 -0.49491   1.00041

__Is our covariance matrix estimate close?__ In this case (since we generated the data ourselves), we can compare our empirical covariance matrix estimate `Σ̂::Array{Float64,2}` to the true covariance matrix `Σ::Array{Float64,2}` used to generate the synthetic data. 

> __Test:__ We'll compare the two covariance matrices by computing the Frobenius norm of their difference. The Frobenius norm of a matrix $\mathbf{A} \in \mathbb{R}^{n \times m}$ is defined as:
> $$
\|\mathbf{A}\|_{F} = \sqrt{\sum_{i=1}^{n}\sum_{j=1}^{m} |a_{ij}|^{2}}
> $$
> where $a_{ij}$ is the element in the $i^{th}$ row and $j^{th}$ column of matrix $\mathbf{A}$. If the Frobenius norm of the difference between the two covariance matrices is very small (close to zero), it indicates that they are nearly identical, confirming the correctness of our implementation. We'll use the [`@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) to enforce this check.

What do we get?

In [9]:
let

    # initialize -
    ϵ = 1e-1; # tolerance for the Frobenius norm comparison
    Δ = Σ̂ - Σ; # difference between the estimated and true covariance matrices
    frobenius_norm = norm(Δ); # Frobenius norm (default for matrices)
    test = frobenius_norm < ϵ

    # if test fails, throw an error -
    @assert test "Covariance matrices do not match within tolerance!"
end

If we get here, then our covariance matrix estimate is approximately equal to the true covariance matrix! Let's decompose the covariance matrix using eigendecomposition to get the eigenvalues and eigenvectors.

We'll save the eigenvalues in the `λ̂::Array{Float64,1}` variable and the eigenvectors in the `V̂::Array{Float64,2}` variables. The eigenvalues and eigenvectors are sorted in descending order based on the eigenvalue magnitude. Let's compute the eigendecomposition using the built-in Julia [`eigen(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen). 

In [10]:
(λ̂, V̂) = let

    # compute the eigendecomposition using the built-in function -
    F = eigen(Σ̂);
    λ = F.values; # grab the eigenvalues
    V = F.vectors; # grab the eigenvectors

    # sort the eigenpairs by eigenvalue magnitude -
    p = sortperm(λ, rev=true); # indices that would sort λ in descending order
    λ = λ[p]
    V = V[:,p]

    (λ, V) # return the eigenvalues and eigenvectors
end

([1.496719413142198, 0.506895637841614], [-0.7081011322106272 -0.7061110299110388; 0.7061110299110388 -0.7081011322106272])

__Singular values vs. Eigenvalues Test:__ Finally, let's verify the relationship between the singular values and the eigenvalues of the covariance matrix. Specifically, we will check if each eigenvalue `λ̂[i]` is approximately equal to the square of the corresponding singular value `S[i]` divided by `(n-1)`, where `n` is the number of samples in our dataset.

In [11]:
let
   
    ϵ = 1e-4; # tolerance for approximate equality check
    n = size(X, 1); # number of samples
    for i in 1:length(λ̂)
        @assert isapprox(λ̂[i], (S[i]^2)/(n-1); atol=ϵ) "Eigenvalue $(i) is not approximately equal to the square of the corresponding singular value divided by (n-1)!"
    end
end

___

## Task 2: What Does It All Mean?
In this final task, let's interpret the results of our eigenvalue and eigenvector computations in the context of our synthetic dataset.

> __What do eigenvalues and eigenvectors represent?__ In the context of our synthetic dataset, the dominant eigenvalue $\lambda_1$ represents the amount of variance captured along the direction of the dominant eigenvector $\mathbf{v}_1$. The eigenvector $\mathbf{v}_1$ indicates the direction in the feature space along which the data varies the most.

Is that what we see?

In [13]:
let

    # initialize -
    zscore = 3.91; # z-score for 99.99% confidence interval
    scatter(X[:,1], X[:,2], label="", c=:gray67,msc=:gray50) # plot the data points

    # plot coordinate axes through the origin
    vline!([0.0], color=:gray, lw=1, ls=:dot, label="")
    hline!([0.0], color=:gray, lw=1, ls=:dot, label="")
    scatter!([0], [0], color=:black, ms=4, label="")

    # overlay the leading eigenvector as an arrow (sign is arbitrary)
    μ = vec(mean(X, dims=1))
    scatter!([μ[1]], [μ[2]], color=:black, ms=4, label="")
    scale = zscore * sqrt(λ₁)
    x0, y0 = μ

    # +v₁ direction
    x1 = x0 + scale * v₁[1]
    y1 = y0 + scale * v₁[2]
    plot!([x0, x1], [y0, y1], arrow=:arrow, lw=3, color=:red, label="v₁")

    # -v₁ direction (same eigenspace)
    x2 = x0 - scale * v₁[1]
    y2 = y0 - scale * v₁[2]
    plot!([x0, x2], [y0, y2], arrow=:arrow, lw=2, color=:red, ls=:dash, label="-v₁")
    

    # background, and labels
    plot!(bg="gray95", background_color_outside="white", framestyle = :box, fg_legend = :transparent);
    xlabel!("Feature 1", fontsize=18)
    ylabel!("Feature 2", fontsize=18)
end


UndefVarError: UndefVarError: `λ₁` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

__Why do we include the negative direction?__
When we estimate the eigenvector $\mathbf{v}_1$, both $\mathbf{v}_1$ and $-\mathbf{v}_1$ are valid eigenvectors corresponding to the same eigenvalue $\lambda_1$. This is because eigenvectors are defined up to a sign; if $\mathbf{v}$ is an eigenvector, then so is $-\mathbf{v}$. 

Suppose $\mathbf{A}\mathbf{v} = \lambda \mathbf{v}$. Multiplying both sides by -1 gives $\mathbf{A}(-\mathbf{v}) = -\lambda \mathbf{v} = \lambda(-\mathbf{v})$, confirming that $-\mathbf{v}$ is also an eigenvector associated with the eigenvalue $\lambda$.

___

## Summary
We generated synthetic two-dimensional data, computed its empirical covariance matrix, and used power iteration to find the dominant eigenvalue and eigenvector.

> __Key Takeaways:__
> 
> * **Empirical covariance captures data spread:** The sample covariance matrix $\hat{\mathbf{\Sigma}}$ quantifies how variables vary together and is computed by centering data and normalizing by $n-1$
> * **Power iteration finds dominant directions:** The iterative method converges to the largest eigenvalue and its eigenvector without computing the full eigendecomposition
> * **Dominant eigenvectors indicate maximum variance:** The eigenvector corresponding to the largest eigenvalue points in the direction where the data exhibits the greatest spread


These techniques form the foundation for dimensionality reduction methods like Principal Component Analysis.
___